# Notebook 01 — Data Ingestion & Preprocessing

**Goal**: Load EuroVerdict, PolyTruth, and Hugging Face (`rares127/dezinformare-ro`) datasets,
clean and standardise them, then create stratified 80/10/10 train/val/test splits.

**Output schema**: `claim_id`, `claim_text`, `evidence_text`, `veracity_label`, `justification`

Splits saved to `data/processed/`.

In [58]:
# Install dependencies (run once)
# %pip install datasets pandas scikit-learn beautifulsoup4 requests tqdm

In [59]:
import re
import uuid
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
# Pre-create all output directories so later notebooks never hit a missing-dir error
for _d in [PROCESSED_DIR, RAW_DIR,
           Path('../data/models'),
           Path('../data/figures')]:
    _d.mkdir(parents=True, exist_ok=True)

LABEL_MAP_NORM = {
    # EuroVerdict labels
    'true': 'true',
    'adevarat': 'true',
    'adevărat': 'true',
    'false': 'false',
    'fals': 'false',
    'context lipsă': 'partially_true',
    'context lipsa': 'partially_true',
    'falsa': 'false',
    'parțial fals': 'partially_true',
    'partial fals': 'partially_true',
    'partially false': 'partially_true',
    'dezinformare_pro_rusa': 'false',
    'stire_credibila': 'true',
    
    'partially true': 'partially_true',
    'partial true': 'partially_true',
    'parțial adevărat': 'partially_true',
    'partial adevarat': 'partially_true',
    'misleading': 'partially_true',
    'manipulare': 'partially_true',
    'manipulari': 'partially_true',
    'manipulat': 'partially_true',
    'trunchiat': 'partially_true',
    'fals partial': 'partially_true',
    'înşelător': 'partially_true',
    # PolyTruth / dezinformare-ro labels
    '0': 'false',
    '1': 'true',
    '2': 'partially_true',
}

print('Paths ready.')

Paths ready.


## 1. Cleaning Utilities

In [60]:
def remove_html(text: str) -> str:
    """Strip HTML tags using a simple regex (no external dep needed)."""
    return re.sub(r'<[^>]+>', ' ', str(text)).strip()


# Romanian diacritic standardisation:
# Normalise ş→ș, ţ→ț (cedilla → comma-below, which is standard modern Romanian)
_DIACRITIC_MAP = str.maketrans(
    'şŞţŢ',   # cedilla variants
    'șȘțȚ',   # comma-below variants
)

def normalise_diacritics(text: str) -> str:
    text = text.translate(_DIACRITIC_MAP)
    # Also decompose + recompose to handle Unicode edge cases
    text = unicodedata.normalize('NFC', text)
    return text


def clean_text(text: str) -> str:
    text = remove_html(text)
    text = normalise_diacritics(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalise_label(raw: str) -> str:
    key = str(raw).lower().strip()
    return LABEL_MAP_NORM.get(key, 'unknown')


print('Cleaning utilities defined.')

Cleaning utilities defined.


## 2. Load EuroVerdict (Romanian subset)

In [61]:
import pandas as pd

# The official dataset is hosted on GitHub, not Hugging Face
url = "https://raw.githubusercontent.com/LanD-FBK/EuroVerdict/main/data/EuroVerdict_Dataset.json"

try:
    # Load the JSON file directly into Pandas
    euro_df = pd.read_json(url)
    print(f'EuroVerdict full shape: {euro_df.shape}')

    # Filter for the Romanian language subset ('ro')
    euro_ro_df = euro_df[euro_df['Language'] == 'ro']
    print(f'Romanian split shape: {euro_ro_df.shape}')
    print(euro_ro_df.columns.tolist())

except Exception as e:
    print(f'Could not load EuroVerdict from GitHub: {e}')
    print('Attempting to load from local file data/raw/euroverdict_ro.csv …')
    # euro_df = pd.read_csv(RAW_DIR / 'euroverdict_ro.csv')

EuroVerdict full shape: (1642, 11)
Romanian split shape: (192, 11)
['ID', 'Language', 'Date', 'Publisher', 'Publisher_Website', 'Claim', 'Verdict', 'Rating', 'Article_Url', 'External_Evidence', 'Set']


In [62]:
# Filter Romanian rows if a language column exists
lang_col = next((c for c in euro_df.columns if 'lang' in c.lower()), None)
if lang_col:
    euro_df = euro_df[euro_df[lang_col].str.lower().str.startswith('ro')].copy()
    print(f'After Romanian filter: {euro_df.shape}')

# Map to standard schema — adjust column names to actual dataset columns
EURO_COL_MAP = {
    'Claim': 'claim_text',
    'External_Evidence': 'evidence_text',
    'Rating': 'veracity_label',
    'Verdict': 'justification',
}

rename_dict = {k: v for k, v in EURO_COL_MAP.items() if k in euro_df.columns}
euro_df = euro_df.rename(columns=rename_dict)
euro_df['evidence_text'] = euro_df.get('evidence_text', pd.Series(dtype=str)).apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in euro_df.columns:
        euro_df[col] = ''

euro_df['source'] = 'euroverdict'
euro_df['claim_id'] = [f'ev_{uuid.uuid4().hex[:8]}' for _ in range(len(euro_df))]

print(euro_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

After Romanian filter: (192, 11)
        claim_id                                         claim_text  \
833  ev_0aaa8a60  Acest videoclip o arată pe Nicole Schwab cerân...   
834  ev_e1e1336e  Această fotografie arată un vaccin cu vaporiza...   
835  ev_28c2016a  Speakerul de la Davos, Damon Imani, l-a insult...   

    veracity_label  
833           Fals  
834           Fals  
835           Fals  


## 3. Load PolyTruth (Romanian subset)

In [63]:
import pandas as pd
print('Skipping PolyTruth as requested by user.')
poly_df = pd.DataFrame(columns=['claim_id', 'claim_text', 'evidence_text', 'veracity_label', 'justification', 'source'])

Skipping PolyTruth as requested by user.


In [64]:
POLY_COL_MAP = {
    'statement':    'claim_text',
    'correction':   'justification',
    'label':        'veracity_label',
    'context':      'evidence_text',
}
rename_dict = {k: v for k, v in POLY_COL_MAP.items() if k in poly_df.columns}
poly_df = poly_df.rename(columns=rename_dict)

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in poly_df.columns:
        poly_df[col] = ''

poly_df['source'] = 'polytruth'
poly_df['claim_id'] = [f'pt_{uuid.uuid4().hex[:8]}' for _ in range(len(poly_df))]

print(poly_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

Empty DataFrame
Columns: [claim_id, claim_text, veracity_label]
Index: []


## 4. Load dezinformare-ro (rares127/dezinformare-ro)

In [65]:
from datasets import load_dataset
try:
    dezinfo_ds = load_dataset('rares127/dezinformare-ro', split='train')
    dezinfo_df = dezinfo_ds.to_pandas()
    print(f'dezinformare-ro raw shape: {dezinfo_df.shape}')
    print(dezinfo_df.columns.tolist())
except Exception as e:
    print(f'Could not load dezinformare-ro: {e}')
    dezinfo_df = pd.read_csv(RAW_DIR / 'dezinformare_ro.csv')

dezinformare-ro raw shape: (1483, 18)
['id', 'url', 'titlu', 'data', 'an', 'luna', 'sursa_site', 'sectiune', 'text_curat', 'stire_citata', 'naratiuni_false', 'obiective_propaganda', 'nr_cuvinte_v4', 'nr_cuvinte_truncat', 'calitate_extractie', 'label', 'label_numeric', 'hash_continut']


In [66]:
DEZINFO_COL_MAP = {
    'titlu': 'claim_text',
    'text_curat': 'evidence_text',
    'label': 'veracity_label',
    'naratiuni_false': 'justification',
}
rename_dict = {k: v for k, v in DEZINFO_COL_MAP.items() if k in dezinfo_df.columns}
dezinfo_df = dezinfo_df.rename(columns=rename_dict)

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in dezinfo_df.columns:
        dezinfo_df[col] = ''

dezinfo_df['source'] = 'dezinformare-ro'
dezinfo_df['claim_id'] = [f'dz_{uuid.uuid4().hex[:8]}' for _ in range(len(dezinfo_df))]

print(dezinfo_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

      claim_id                                         claim_text  \
0  dz_fbb6cfa6  PROPAGANDĂ DE RĂZBOI: Masacrul de la Izium a f...   
1  dz_62a8eb03                Rusia a invadat România doar o dată   
2  dz_a2244d59  Imagini șocante: Militarii ruși execută „Ordin...   

          veracity_label  
0  dezinformare_pro_rusa  
1  dezinformare_pro_rusa  
2        stire_credibila  


## 5. Optional: Load stopfals.md scraped data

In [67]:
import json

stopfals_path = RAW_DIR / 'stopfals.jsonl'
if stopfals_path.exists():
    records = [json.loads(l) for l in stopfals_path.read_text(encoding='utf-8').splitlines() if l]
    sf_df = pd.DataFrame(records)
    sf_df = sf_df.rename(columns={'verdict': 'veracity_label'})
    for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
        if col not in sf_df.columns:
            sf_df[col] = ''
    sf_df['source'] = 'stopfals'
    sf_df['claim_id'] = [f'sf_{uuid.uuid4().hex[:8]}' for _ in range(len(sf_df))]
    print(f'stopfals.md records: {len(sf_df)}')
else:
    print('No stopfals.jsonl found — run src/scraper.py if more data is needed.')
    sf_df = pd.DataFrame(columns=['claim_id', 'claim_text', 'evidence_text', 'veracity_label', 'justification', 'source'])

stopfals.md records: 1620


## 6. Merge, Clean & Standardise

In [68]:
KEEP_COLS = ['claim_id', 'claim_text', 'evidence_text', 'veracity_label', 'justification', 'source']

frames = []
for df in [euro_df, poly_df, dezinfo_df, sf_df]:
    sub = df[[c for c in KEEP_COLS if c in df.columns]].copy()
    for c in KEEP_COLS:
        if c not in sub.columns:
            sub[c] = ''
    frames.append(sub)

combined = pd.concat(frames, ignore_index=True)
print(f'Combined shape before cleaning: {combined.shape}')

Combined shape before cleaning: (3295, 6)


In [69]:
# Apply text cleaning
for col in ['claim_text', 'evidence_text', 'justification']:
    combined[col] = combined[col].fillna('').apply(clean_text)

# Normalise labels
combined['veracity_label'] = combined['veracity_label'].apply(normalise_label)

# Drop rows with unknown labels or empty claim text
before = len(combined)
print(f'Before cleaning: {before}')
combined = combined[combined['veracity_label'] != 'unknown']
print(f'After cleaning: {len(combined)}')
combined = combined[combined['claim_text'].str.len() > 10]
print(f'After cleaning: {len(combined)}')
combined = combined.drop_duplicates(subset='claim_text')
print(f'After dropping: {len(combined)}')
print(f'Dropped {before - len(combined)} rows (unknown labels / empty claims / duplicates)')
print(f'Final dataset shape: {combined.shape}')
print(combined['veracity_label'].value_counts())

Before cleaning: 3295
After cleaning: 2602
After cleaning: 2602
After dropping: 1683
Dropped 1612 rows (unknown labels / empty claims / duplicates)
Final dataset shape: (1683, 6)
veracity_label
false             925
true              737
partially_true     21
Name: count, dtype: int64


## 7. Stratified 80/10/10 Split

No article text overlap across splits to prevent data leakage.

In [70]:
from sklearn.model_selection import train_test_split

combined = combined.reset_index(drop=True)

train_df, temp_df = train_test_split(
    combined,
    test_size=0.20,
    stratify=combined['veracity_label'],
    random_state=42,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['veracity_label'],
    random_state=42,
)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}')
print('Train label dist:')
print(train_df['veracity_label'].value_counts())

# Verify no evidence_text overlap between train and test
train_evidence = set(train_df['evidence_text'].dropna())
test_overlap = test_df['evidence_text'].isin(train_evidence).sum()
print(f'Evidence text overlap (train ∩ test): {test_overlap} rows')

Train : 1346 | Val : 168 | Test : 169
Train label dist:
veracity_label
false             740
true              589
partially_true     17
Name: count, dtype: int64
Evidence text overlap (train ∩ test): 4 rows


In [71]:
train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False, encoding='utf-8')
val_df.to_csv(PROCESSED_DIR / 'val.csv', index=False, encoding='utf-8')
test_df.to_csv(PROCESSED_DIR / 'test.csv', index=False, encoding='utf-8')

print('Saved:')
print(f'  {PROCESSED_DIR}/train.csv ({len(train_df)} rows)')
print(f'  {PROCESSED_DIR}/val.csv   ({len(val_df)} rows)')
print(f'  {PROCESSED_DIR}/test.csv  ({len(test_df)} rows)')

Saved:
  ..\data\processed/train.csv (1346 rows)
  ..\data\processed/val.csv   (168 rows)
  ..\data\processed/test.csv  (169 rows)
